# Helm release lifecycle drill

Hands-on walkthrough of the helm command surface for managing a release through install, upgrade, rollback, and uninstall, with helm history used to inspect every state change.

## Purpose

Helm charts package Kubernetes manifests into reusable, versioned releases. A team needs to understand the full lifecycle — install a chart, upgrade it with new configuration, roll back when an upgrade misbehaves, and clean up when done — and inspect the history that ties each step together.

## Setup

This drill uses a local Kubernetes cluster (minikube or kind) and a lightweight chart. The chart below is written to a scratch directory so no external repo is required.

In [ ]:
import os
import subprocess
import json
from pathlib import Path

scratch = Path("/tmp/helm-lifecycle-drill")
scratch.mkdir(parents=True, exist_ok=True)
os.chdir(str(scratch))
print(f"Working in {scratch}")

In [ ]:
result = subprocess.run(["helm", "version"], capture_output=True, text=True)
if result.returncode != 0:
    print("Helm not found — install it first")
else:
    print(result.stdout.strip())

## Step 1 — Build a minimal chart

A chart with one Deployment and one Service, written as plain YAML files under the scratch directory.

In [ ]:
(scratch / "charts").mkdir(exist_ok=True)
chart_dir = scratch / "charts" / "webapp"
(chart_dir / "templates").mkdir(parents=True)

(chart_dir / "Chart.yaml").write_text("""apiVersion: v2
name: webapp
version: 0.1.0
appVersion: \"1.0\"
""")

(chart_dir / "values.yaml").write_text("""replicaCount: 1
image:
  repository: nginx
  tag: \"1.25\"
  pullPolicy: IfNotPresent
service:
  type: ClusterIP
  port: 80
""")

(chart_dir / "templates" / "deployment.yaml").write_text("""apiVersion: apps/v1
kind: Deployment
metadata:
  name: webapp
spec:
  replicas: {{ .Values.replicaCount }}
  selector:
    matchLabels:
      app: webapp
  template:
    metadata:
      labels:
        app: webapp
    spec:
      containers:
      - name: webapp
        image: "{{ .Values.image.repository }}:{{ .Values.image.tag }}"
        ports:
        - containerPort: {{ .Values.service.port }}
""")

(chart_dir / "templates" / "service.yaml").write_text("""apiVersion: v1
kind: Service
metadata:
  name: webapp-svc
spec:
  type: {{ .Values.service.type }}
  selector:
    app: webapp
  ports:
  - port: {{ .Values.service.port }}
    targetPort: {{ .Values.service.port }}
""")

print(f"Chart written to {chart_dir}")

## Step 2 — Install the release

Use `helm install` to deploy the chart to the cluster with a release name. The release is versioned from the start.

In [ ]:
result = subprocess.run(
    ["helm", "install", "webapp-rel", str(chart_dir), "--dry-run", "--debug"],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print("Install rendered successfully (dry-run)")
else:
    print("Render failed — check chart syntax")
    print(result.stderr[-500:] if result.stderr else "")

## Step 3 — Upgrade the release

Change the replica count and image tag via `helm upgrade`. The release history records each upgrade as a new revision.

In [ ]:
values_v2 = """replicaCount: 2
image:
  repository: nginx
  tag: \"1.26\"
  pullPolicy: IfNotPresent
service:
  type: ClusterIP
  port: 80
"""
(scratch / "values-v2.yaml").write_text(values_v2)
print("Wrote values-v2.yaml with updated replicas and image tag")

In [ ]:
result = subprocess.run(
    ["helm", "upgrade", "webapp-rel", str(chart_dir), "-f", str(scratch / "values-v2.yaml"), "--dry-run", "--debug"],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print("Upgrade rendered successfully (dry-run)")
else:
    print("Upgrade render failed")
    print(result.stderr[-500:] if result.stderr else "")

## Step 4 — Inspect release history

Run `helm history` to see every revision of the release, including the install (revision 1) and the upgrade (revision 2).

In [ ]:
result = subprocess.run(
    ["helm", "history", "webapp-rel", "--all"],
    capture_output=True, text=True,
)
print(result.stdout if result.returncode == 0 else result.stderr)

## Step 5 — Rollback a release

If an upgrade introduces a problem, `helm rollback` reverts the release to a previous revision. Helm creates a new revision for the rollback itself.

In [ ]:
result = subprocess.run(
    ["helm", "rollback", "webapp-rel", "1", "--dry-run", "--debug"],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print("Rollback rendered successfully (dry-run)")
else:
    print("Rollback render failed")
    print(result.stderr[-500:] if result.stderr else "")

## Step 6 — Uninstall the release

`helm uninstall` removes all resources managed by the release. The release history is preserved — `helm list --all` still shows it.

In [ ]:
result = subprocess.run(
    ["helm", "uninstall", "webapp-rel", "--dry-run"],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print("Uninstall rendered successfully (dry-run)")
else:
    print("Uninstall render failed")
    print(result.stderr[-500:] if result.stderr else "")

## Verify

Key things to confirm after each step:

- `helm install` creates a new release with revision 1
- `helm upgrade` increments the revision (2, 3, ...)
- `helm history` shows all revisions with their status (deployed, failed, superseded)
- `helm rollback` creates a new revision that mirrors a previous one
- `helm uninstall` removes managed resources but keeps release history visible via `helm list --all`

The dry-run flags used above let the drill run without a live cluster while still validating chart syntax and rendering.

## What I'd try next

Run the full lifecycle against a real cluster (drop the dry-run flags), then explore `helm template` for local rendering and `helm diff` for previewing changes between revisions.